In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%run cut_mda.py --in_root ../Data --out_root ../Data/mda_output

2026-02-15 15:06:22,408 - INFO - Found 42 files to process
2026-02-15 15:06:22,408 - INFO - Mode: accurate, Workers: 16
Extracting MD&A: 100%|██████████| 42/42 [00:24<00:00,  1.73it/s, success=42]
2026-02-15 15:06:46,868 - INFO - ============================================================
2026-02-15 15:06:46,868 - INFO - EXTRACTION COMPLETE
2026-02-15 15:06:46,868 - INFO - ============================================================
2026-02-15 15:06:46,869 - INFO - Total files: 42
2026-02-15 15:06:46,869 - INFO - Success: 42 (100.0%)
2026-02-15 15:06:46,870 - INFO - Failed: 0
2026-02-15 15:06:46,870 - INFO - Skipped: 0
2026-02-15 15:06:46,870 - INFO - Errors: 0
2026-02-15 15:06:46,871 - INFO - Extraction methods used: {'dom': 42}
2026-02-15 15:06:46,871 - INFO - Metadata saved to: ../Data/mda_output/mda_metadata/extraction_metadata_20260215_150646.jsonl
2026-02-15 15:06:46,871 - INFO - Summary saved to: ../Data/mda_output/mda_metadata/extraction_summary_20260215_150646.json


# Factor Extraction

In [7]:
# =========================================================================================
#  MD&A Factor Extraction Runner — Filesystem + Parallel Version (2025-08-23)
#  Purpose:
#    A fully automated pipeline to extract structured factors from MD&A sections
#    of SEC 10-K/10-Q filings using HTML-based chunking, OpenAI function calling,
#    and parallel filesystem orchestration.
#
#  Key Features:
#  1. **Filesystem-based Workflow** — Reads cleaned MD&A HTML from `sec_data/mda_output/mda_clean/`,
#     writes intermediates to `factors/intermediate/{TICKER}/`, and final factor JSONs to
#     `factors/mda_factors/{TICKER}/`.
#  2. **Parallel Processing** — Multi-threaded execution:
#        - multiple filings (`MAX_WORKERS_REPORTS`)
#        - multiple chunks per filing (`MAX_WORKERS_CHUNKS`)
#        - bounded concurrent OpenAI calls (`MAX_IN_FLIGHT_OPENAI`)
#  3. **Two-Stage Extraction** —
#        (a) Chunk-level question answering with evidence gating (`EXTRACT_MODEL`)
#        (b) Cross-chunk synthesis into consolidated factor summaries (`SYNTH_MODEL`)
#  4. **Strict JSON Schema** — Tool-call responses validated against strict parameter
#     schemas for reproducibility and future versioning (`SCHEMA_VERSION`).
#  5. **Resume-Safe + Idempotent** — Skips already-processed chunks/reports,
#     retries transient errors with exponential backoff, and preserves per-chunk outputs.
#  6. **Configurable Granularity** — Token budgets, overlap, max factors/evidence/notes,
#     temperature, and backoff all tunable in header constants.
#  7. **Human-Readable Trace** — Clear `[chunk]`, `[done]`, `[skip-final-exists]` logging
#     to monitor progress.
#
#  Output:
#    Each processed filing → `{TICKER}_{MM-DD-YYYY}_{FORM}_FACTORS_DETAILED.json`
#    containing a structured list of extracted factors, detailed summaries,
#    and sentiment/impact rationales.
#
#  Run Example:
#      export OPENAI_API_KEY=sk-...
#      python mda_factor_runner_fs_parallel.py
#
#  Schema Version: v2-strict-fs-par-2025-08-23
# =========================================================================================


import os, re, json, time, hashlib, random, threading
from pathlib import Path
from typing import Any, Dict, List, Tuple
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup, Tag, NavigableString

# Local implementations (avoid htmlrag/faiss-cpu build issues)
def _bs4_parser():
    try:
        import lxml
        return "lxml"
    except ImportError:
        return "html.parser"

def clean_html(html: str) -> str:
    """Remove scripts, styles; simplify structure. Compatible with htmlrag interface."""
    soup = BeautifulSoup(html, _bs4_parser())
    for tag in soup.find_all(["script", "style", "noscript", "iframe", "svg"]):
        tag.decompose()
    body = soup.find("body") or soup
    return str(body) if body else str(soup)

def _word_count(s: str) -> int:
    return len(re.findall(r"\S+", s))

def build_block_tree(html: str, max_node_words: int = 256) -> Tuple[List[Tuple[str, List[str], bool]], str]:
    """Build block tree: [(content, path, is_leaf), ...]. Compatible with htmlrag interface."""
    soup = BeautifulSoup(html, _bs4_parser())
    blocks: List[Tuple[str, List[str], bool]] = []

    def _walk(elem, path: List[str]) -> None:
        if elem is None or (hasattr(elem, "name") and elem.name in ("script", "style")):
            return
        tag_children = [c for c in elem.children if hasattr(c, "name") and c.name]
        full_text = (elem.get_text() if hasattr(elem, "get_text") else str(elem)) or ""
        wc = _word_count(full_text.strip())
        if not tag_children:
            if wc > 0:
                blocks.append((str(elem), path.copy(), True))
            return
        if wc <= max_node_words and wc > 0:
            blocks.append((str(elem), path.copy(), True))
            return
        for child in tag_children:
            _walk(child, path + [getattr(child, "name", "?")])

    root = soup.find("body") or soup.find("html") or soup
    if root:
        _walk(root, [getattr(root, "name", "body")])
    if not blocks:
        blocks = [(html, ["html"], True)]
    return blocks, html

from bs4 import Tag, NavigableString
from openai import OpenAI
import argparse, re

# ---------- Load .env (API key from sample_code/.env; cwd = notebook dir) ----------
for p in [Path(".env"), Path("sample_code/.env")]:
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip('"').strip("'")
        break

# ---------- USER CONFIG ----------
BASE_DIR         = Path("../Data/mda_output/mda_clean")   # Input: cleaned HTML (read-only)
FINAL_JSON_DIR   = Path("factors/mda_factors")            # Output: consolidated factor results (write here only)
INTERMEDIATE_DIR = Path("factors/intermediate")           # All intermediates go here: factors/intermediate/{TICKER}/
QUESTIONS_JSON   = Path("questions.json")               # Question list (same dir as notebook)

# Singleton client (initialized in MAIN)
CLIENT = None

TICKERS_FILTER = {"AAL"}  # Only AAL ticker for factor extraction
START_YEAR               = 2016
MAX_REPORTS_TOTAL        = 1   # None = no global limit
MAX_REPORTS_PER_TICKER   = None   # None = no per-ticker limit

# Models (local GPT OSS)
EXTRACT_MODEL = "openai/gpt-oss-20b"   # per-chunk extraction
SYNTH_MODEL   = "openai/gpt-oss-20b"   # cross-chunk synthesis

# Extraction + synthesis controls
EVIDENCE_MIN_WORDS       = 6
SUMMARY_MAX_CHARS        = 240
EVIDENCE_MAX_CHARS       = 280
MAX_ANSWERS_ITEMS        = 140

MAX_FACTORS_PER_BATCH    = 18
MAX_EVIDENCE_PER_FACTOR  = 8
MAX_EVIDENCE_CHARS       = 320
MAX_NOTES_PER_FACTOR     = 12
MAX_NOTES_CHARS          = 240
DETAILED_SUMMARY_MAX     = 1600
IMPACT_RATIONALE_MAX     = 900

TEMPERATURE              = 0
PAUSE_BETWEEN_CALLS      = 0.10
SCHEMA_VERSION           = "v2-strict-fs-par-2025-08-23"

# Parallelism knobs (reduced for local GPT OSS to avoid overloading)
MAX_WORKERS_REPORTS      = 2        # process multiple reports in parallel
MAX_WORKERS_CHUNKS       = 10       # process multiple chunks per report in parallel
MAX_IN_FLIGHT_OPENAI     = 2        # additional cap on concurrent OpenAI calls
MAX_RETRIES              = 5
BASE_BACKOFF             = 0.75

# ------------- tiny utils -------------
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def coerce_html(x: Any) -> str:
    if x is None: return ""
    if isinstance(x, (Tag, NavigableString)): return str(x)
    if isinstance(x, bytes):
        try: return x.decode("utf-8","ignore")
        except Exception: return str(x)
    return str(x)

def safe_tokenizer():
    import tiktoken
    try:
        return tiktoken.get_encoding("cl100k_base")
    except Exception:
        from tiktoken import encoding_for_model
        try:
            return encoding_for_model("gpt-4o")
        except Exception:
            class _Proxy:
                def encode(self, s): return (s if isinstance(s,str) else str(s)).encode("utf-8","ignore")
                def decode(self, b): return b.decode("utf-8","ignore")
            return _Proxy()

def is_meaningful_html(s) -> bool:
    html = coerce_html(s)
    text = re.sub(r"<[^>]+>", "", html, flags=re.DOTALL)
    return bool(text.strip())

def normalize_block_tuple(b: Any) -> Tuple[str, Any, bool]:
    if isinstance(b, (tuple, list)) and len(b) >= 3:
        return coerce_html(b[0]), b[1], bool(b[2])
    if hasattr(b, "get"):
        html_cand = b.get("html") or b.get("node") or b.get("content") or b.get("block_html") or ""
        return coerce_html(html_cand), b.get("path", []), bool(b.get("is_leaf", b.get("leaf", True)))
    return coerce_html(b), [], True

def make_run_dir(html_path: Path) -> Path:
    """
    Intermediate directory:
      factors/intermediate/{TICKER}/
    Filenames include the original stem prefix to avoid collisions.
    """
    ticker = (html_path.name.split("_")[0] if "_" in html_path.name else html_path.name.split("-")[0]).upper()
    run_dir = INTERMEDIATE_DIR / ticker
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def split_batches(tickers, batch_size=10):
    batches = {}
    for i in range(0, len(tickers), batch_size):
        batch_id = i // batch_size + 1   # 1,2,3,...
        batches[batch_id] = tickers[i:i+batch_size]
    return batches

def parse_batch_selector(selector: str) -> list[int]:
    """'3,5-7' -> [3,5,6,7]; '4' -> [4]"""
    out = set()
    for part in selector.split(","):
        part = part.strip()
        if re.fullmatch(r"\d+", part):
            out.add(int(part))
        else:
            m = re.fullmatch(r"(\d+)-(\d+)", part)
            if not m: raise ValueError(f"Bad batch selector: {part}")
            a, b = int(m.group(1)), int(m.group(2))
            if a > b: a, b = b, a
            out.update(range(a, b+1))
    return sorted(out)

# ------------- chunking -------------
def split_block_if_oversized(block_html: Any, max_tokens: int, enc) -> List[str]:
    b = coerce_html(block_html)
    if len(enc.encode(b)) <= max_tokens:
        return [b]
    pieces = re.split(r"(</(?:p|li|tr|div|section|h[1-6]|tbody|thead|table|ul|ol|dl)>)", b, flags=re.I)
    merged, buf = [], ""
    for i in range(0, len(pieces), 2):
        seg = pieces[i]
        close = pieces[i+1] if i+1 < len(pieces) else ""
        cand = buf + seg + close
        if not cand.strip(): continue
        if len(enc.encode(cand)) <= max_tokens:
            buf = cand
        else:
            if buf: merged.append(buf)
            cand2 = seg + close
            if len(enc.encode(cand2)) > max_tokens:
                toks = enc.encode(cand2)
                for j in range(0, len(toks), max_tokens):
                    try: merged.append(enc.decode(toks[j:j+max_tokens]))
                    except Exception: merged.append(coerce_html(cand2)[j:j+max_tokens])
                buf = ""
            else:
                buf = cand2
    if buf: merged.append(buf)
    if not merged:
        toks = enc.encode(b)
        for j in range(0, len(toks), max_tokens):
            try: merged.append(enc.decode(toks[j:j+max_tokens]))
            except Exception: merged.append(coerce_html(b)[j:j+max_tokens])
    return merged

def pack_blocks_into_chunks(block_tree: List[Any], budget_tokens: int, enc, overlap_blocks: int = 1) -> List[str]:
    leaves: List[str] = []
    for node in block_tree:
        html, path, is_leaf = normalize_block_tuple(node)
        if is_leaf and is_meaningful_html(html):
            leaves.append(html)

    flat: List[str] = []
    for html in leaves:
        for p in split_block_if_oversized(html, budget_tokens, enc):
            if is_meaningful_html(p): flat.append(p)

    chunks, cur, cur_tokens = [], [], 0

    def flush():
        nonlocal cur, cur_tokens
        if cur:
            body = "\n".join(cur)
            chunk_html = "<html><body>\n" + body + "\n</body></html>"
            chunks.append(chunk_html)
            cur, cur_tokens = [], 0

    for block_html in flat:
        t = len(enc.encode(block_html))
        if t > budget_tokens:
            if cur: flush()
            chunks.append("<html><body>\n" + block_html + "\n</body></html>")
            continue
        if cur_tokens + t > budget_tokens and cur:
            tail = cur[-overlap_blocks:] if overlap_blocks > 0 else []
            flush()
            cur.extend(tail)
            cur_tokens = len(enc.encode("\n".join(tail))) if tail else 0
        cur.append(block_html)
        cur_tokens += t
    flush()
    return chunks

# ------------- questions & prompts -------------
def flatten_questions(nested: Dict, prefix: str = "") -> List[Tuple[str, str]]:
    out = []
    for k, v in nested.items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            out.extend(flatten_questions(v, key))
        else:
            out.append((key, str(v)))
    return out

def build_prompt_for_chunk(chunk_html: str, key_qs: List[Tuple[str, str]]) -> str:
    questions_json = [{"key": k, "question": q} for (k, q) in key_qs]
    return (
        "You are an information extraction model. You will receive one HTML CHUNK from a 10-K/10-Q MD&A.\n"
        "Answer a fixed list of questions STRICTLY using only this chunk. Do NOT guess or use prior knowledge.\n\n"
        f"RULES:\n- For each question:\n"
        f"  - Search the chunk for DIRECT textual evidence.\n"
        f"  - If you can cite at least one quote with >= {EVIDENCE_MIN_WORDS} words, set \"found\": true and write a concise neutral \"summary\" (<= {SUMMARY_MAX_CHARS} chars) based only on the quoted evidence.\n"
        f"  - If you cannot find such evidence, set \"found\": false and \"summary\": \"\".\n"
        f"- Include up to 1 short, verbatim evidence quote (<= {EVIDENCE_MAX_CHARS} chars) from the chunk in \"evidence\".\n"
        "- Return ONLY via the provided tool schema.\n\n"
        "----- BEGIN CHUNK HTML -----\n"
        + chunk_html +
        "\n----- END CHUNK HTML -----\n\n"
        "QUESTION LIST (keep order):\n"
        + json.dumps(questions_json, ensure_ascii=False, indent=2)
    )

# ------------- OpenAI calls -------------
def strip_code_fences(text: str) -> str:
    t = (text or "").strip()
    if not t.startswith("```"):
        return t
    first_nl = t.find("\n")
    if first_nl != -1:
        t = t[first_nl + 1 :]
    if "```" in t:
        t = t.rsplit("```", 1)[0]
    return t.strip()

def extract_first_balanced_json(text: str) -> str | None:
    s = text or ""
    start = None
    open_ch = None
    for i, ch in enumerate(s):
        if ch in "{[":
            start = i
            open_ch = ch
            break
    if start is None or open_ch is None:
        return None
    close_ch = "}" if open_ch == "{" else "]"
    depth = 0
    in_str = False
    esc = False
    for j in range(start, len(s)):
        c = s[j]
        if in_str:
            if esc:
                esc = False
                continue
            if c == "\\":
                esc = True
                continue
            if c == '"':
                in_str = False
            continue
        else:
            if c == '"':
                in_str = True
                continue
            if c == open_ch:
                depth += 1
            elif c == close_ch:
                depth -= 1
                if depth == 0:
                    return s[start : j + 1]
    return None

def parse_json_payload(text: str, required_keys: set[str]) -> Dict:
    raw = strip_code_fences(text)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        cand = extract_first_balanced_json(raw)
        if not cand:
            raise RuntimeError(f"No JSON object/array found in content. First 200 chars: {raw[:200]!r}") from None
        try:
            data = json.loads(cand)
        except json.JSONDecodeError as e:
            raise RuntimeError(f"Content JSON is not valid: {e}. First 200 chars: {cand[:200]!r}") from None
    if not isinstance(data, dict):
        raise RuntimeError(f"Expected JSON object, got {type(data).__name__}.")
    missing = [k for k in required_keys if k not in data]
    if missing:
        raise RuntimeError(f"Content JSON missing keys: {missing}. First 200 chars: {raw[:200]!r}")
    return data

def call_extract(client: "OpenAI", model: str, prompt: str, chunk_id: str, raw_debug_path: Path | None = None) -> Dict:
    tools = [{
        "type": "function",
        "function": {
            "name": "record_answers",
            "description": "Return answers for all questions with evidence gating.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {"type": "string", "maxLength": 200},
                    "answers": {
                        "type": "array", "maxItems": MAX_ANSWERS_ITEMS,
                        "items": {
                            "type": "object",
                            "properties": {
                                "key":     {"type": "string",  "maxLength": 200},
                                "found":   {"type": "boolean"},
                                "summary": {"type": "string",  "maxLength": SUMMARY_MAX_CHARS},
                                "evidence": {
                                    "type": "array", "maxItems": 1,
                                    "items": {"type": "string", "maxLength": EVIDENCE_MAX_CHARS}
                                }
                            },
                            "required": ["key", "found", "summary", "evidence"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["chunk_id", "answers"],
                "additionalProperties": False
            }
        }
    }]
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Extract structured answers from HTML; be strict about evidence and format."},
            {"role": "user",   "content": prompt}
        ],
        tools=tools,
        tool_choice={"type": "function", "function": {"name": "record_answers"}},
        parallel_tool_calls=False,
        temperature=TEMPERATURE,
        max_tokens=12000,
    )
    ch = resp.choices[0].message
    if getattr(ch, "tool_calls", None):
        data = json.loads(ch.tool_calls[0].function.arguments)
    else:
        # vLLM may return the tool schema JSON in message.content instead of tool_calls
        text = getattr(ch, "content", None) or ""
        if not text.strip():
            raise RuntimeError("Extract call did not return a tool call or content.")
        try:
            data = parse_json_payload(text, {"chunk_id", "answers"})
        except Exception:
            if raw_debug_path is not None:
                try:
                    raw_debug_path.write_text(text, encoding="utf-8", errors="ignore")
                except Exception:
                    pass
            raise

    answers = data.get("answers", [])
    if not isinstance(answers, list):
        raise RuntimeError("Extract response 'answers' is not a list.")
    out = {"chunk_id": data.get("chunk_id", chunk_id), "answers": []}
    for item in answers:
        if not isinstance(item, dict):
            continue
        key   = item.get("key", "")
        found = bool(item.get("found", False))
        summ  = item.get("summary", "") if found else ""
        evid  = item.get("evidence", [])
        if not isinstance(evid, list):
            evid = [str(evid)]
        out["answers"].append({"key": key, "found": found, "summary": summ, "evidence": evid[:1]})
    if not out["answers"]:
        raise RuntimeError("Extract returned 0 answers; treating as failure to avoid caching empties.")
    return out

def dedup_trim_list(items: List[str], max_items: int, max_chars: int) -> List[str]:
    seen, out = set(), []
    for it in items:
        if not it: continue
        s = str(it).strip()
        if max_chars and len(s) > max_chars: s = s[:max_chars].rstrip() + "…"
        if s in seen: continue
        seen.add(s); out.append(s)
        if len(out) >= max_items: break
    return out

def collect_per_chunk_answers_by_index(index_path: Path) -> List[Dict]:
    idx = json.loads(index_path.read_text(encoding="utf-8"))
    outs = []
    for cp in idx.get("chunks", []):
        ap = Path(cp).with_suffix(".answers.json")
        if ap.exists():
            try:
                d = json.loads(ap.read_text(encoding="utf-8"))
                if isinstance(d, dict) and "answers" in d:
                    outs.append(d)
            except Exception:
                pass
    return outs

def aggregate_by_factor(per_chunk: List[Dict]) -> Dict[str, Dict]:
    merged: Dict[str, Dict] = {}
    for res in per_chunk:
        cid = res.get("chunk_id", "")
        for ans in res.get("answers", []):
            if not ans.get("found"): continue
            k = ans.get("key", "")
            if not k: continue
            m = merged.get(k, {"key": k, "notes": [], "evidence": [], "chunks": []})
            s = ans.get("summary", "")
            if s: m["notes"].append(s)
            for ev in ans.get("evidence", []):
                if ev: m["evidence"].append(ev)
            if cid: m["chunks"].append(cid)
            merged[k] = m
    for k, v in merged.items():
        v["notes"]    = dedup_trim_list(v["notes"],    MAX_NOTES_PER_FACTOR,    MAX_NOTES_CHARS)
        v["evidence"] = dedup_trim_list(v["evidence"], MAX_EVIDENCE_PER_FACTOR, MAX_EVIDENCE_CHARS)
        v["chunks"]   = list(dict.fromkeys(v["chunks"]))
    return merged

def build_synth_prompt(company_hint: str, filing_hint: str, batch_payload: List[Dict]) -> str:
    return (
        "You are a financial analysis assistant. Consolidate factor-level insights across an MD&A filing.\n\n"
        "For EACH factor:\n"
        "1) Read aggregated EVIDENCE quotes and NOTES.\n"
        "2) Write a DETAILED, neutral summary that merges mentions (include specifics; avoid repetition).\n"
        "3) Assess IMPACT: Tailwind, Headwind, Mixed/Depends, or Unclear; add rationale grounded in evidence.\n"
        "4) Do not cite chunk ids. Avoid boilerplate.\n\n"
        f"Company: {company_hint}\nFiling: {filing_hint}\n\n"
        "BATCH INPUT:\n" + json.dumps(batch_payload, ensure_ascii=False, indent=2)
    )

def call_synthesize(client: "OpenAI", model: str, prompt: str, raw_debug_path: Path | None = None) -> Dict:
    tools = [{
        "type": "function",
        "function": {
            "name": "compile_factors",
            "description": "Return detailed, consolidated summaries and impact assessments for each factor.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "maxItems": MAX_FACTORS_PER_BATCH + 5,
                        "items": {
                            "type": "object",
                            "properties": {
                                "factor":           {"type": "string", "maxLength": 200},
                                "detailed_summary": {"type": "string", "maxLength": DETAILED_SUMMARY_MAX},
                                "impact": {
                                    "type": "object",
                                    "properties": {
                                        "classification": {"type": "string", "enum": ["Tailwind", "Headwind", "Mixed/Depends", "Unclear"]},
                                        "rationale":      {"type": "string", "maxLength": IMPACT_RATIONALE_MAX},
                                        "confidence":     {"type": "number", "minimum": 0.0, "maximum": 1.0}
                                    },
                                    "required": ["classification", "rationale", "confidence"],
                                    "additionalProperties": False
                                }
                            },
                            "required": ["factor", "detailed_summary", "impact"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["items"],
                "additionalProperties": False
            }
        }
    }]
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Synthesize factor insights from provided evidence/notes only; mark Unclear if insufficient."},
            {"role": "user",   "content": prompt}
        ],
        tools=tools,
        tool_choice={"type": "function", "function": {"name": "compile_factors"}},
        parallel_tool_calls=False,
        temperature=TEMPERATURE,
        max_tokens=7000,
    )
    ch = resp.choices[0].message
    if getattr(ch, "tool_calls", None):
        data = json.loads(ch.tool_calls[0].function.arguments)
        if not isinstance(data, dict) or "items" not in data:
            raise RuntimeError("Synthesis tool-call JSON missing 'items'.")
        return data

    # Fallback: vLLM may return JSON in content instead of tool call
    text = getattr(ch, "content", None) or ""
    if not text.strip():
        raise RuntimeError("Synthesis call did not return a tool call or content.")
    try:
        data = parse_json_payload(text, {"items"})
        print("[fallback] Parsed synthesis from content (no tool call)")
        return data
    except Exception as e:
        # Save raw content (failure-only)
        if raw_debug_path is not None:
            try:
                raw_debug_path.write_text(text, encoding="utf-8", errors="ignore")
            except Exception:
                pass

        # One repair retry: ask the model to re-emit valid JSON only
        resp2 = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "Return a valid JSON object ONLY. No prose. No code fences. Keys must match the schema."},
                {"role": "user",   "content": prompt},
            ],
            tools=tools,
            tool_choice={"type": "function", "function": {"name": "compile_factors"}},
            parallel_tool_calls=False,
            temperature=TEMPERATURE,
            max_tokens=7000,
        )
        ch2 = resp2.choices[0].message
        if getattr(ch2, "tool_calls", None):
            data2 = json.loads(ch2.tool_calls[0].function.arguments)
            if not isinstance(data2, dict) or "items" not in data2:
                raise RuntimeError("Synthesis repair tool-call JSON missing 'items'.")
            print("[repair] Parsed synthesis from tool call")
            return data2
        text2 = getattr(ch2, "content", None) or ""
        if raw_debug_path is not None:
            try:
                raw_debug_path.with_name(raw_debug_path.stem + "__retry.raw.txt").write_text(text2, encoding="utf-8", errors="ignore")
            except Exception:
                pass
        data2 = parse_json_payload(text2, {"items"})
        print("[repair] Parsed synthesis from content")
        return data2

# ------------- filenames / finder -------------
def parse_date_from_name(name: str):
    m = re.search(r'_(\d{4}-\d{2}-\d{2})_', name)
    if m:
        try: return datetime.strptime(m.group(1), "%Y-%m-%d").date()
        except Exception: pass
    m2 = re.search(r'_(\d{2}-\d{2}-\d{4})_', name)
    if m2:
        try: return datetime.strptime(m2.group(1), "%m-%d-%Y").date()
        except Exception: pass
    return None

def find_reports(base_dir: Path,
                 tickers_filter=None,
                 start_year: int = 2015,
                 limit_total=None,
                 limit_per_ticker=None) -> List[Path]:
    tset = {t.lower() for t in tickers_filter} if tickers_filter else None
    paths: List[Path] = []
    per_ticker_count: Dict[str, int] = {}
    candidates = sorted(base_dir.glob("*_mda.html"))
    for p in candidates:
        base = p.name
        ticker = (base.split("_")[0] if "_" in base else base.split("-")[0]).lower()
        if tset and ticker not in tset: continue
        dt = parse_date_from_name(base)
        if dt and dt.year < start_year: continue
        c = per_ticker_count.get(ticker, 0)
        if limit_per_ticker is not None and c >= limit_per_ticker: continue
        paths.append(p)
        per_ticker_count[ticker] = c + 1
        if limit_total is not None and len(paths) >= limit_total: break
    return paths

def expected_final_json_path(html_path: Path) -> Path:
    """
    Derive the final FACTORS_DETAILED.json path from the HTML source filename.
    Naming rule matches write step:
      {TICKER}_{MM-DD-YYYY}_{FORM}_FACTORS_DETAILED.json
    """
    base = html_path.name
    ticker = (base.split("_")[0] if "_" in base else base.split("-")[0]).upper()
    form = "10-K" if "_10-K_" in base else ("10-Q" if "_10-Q_" in base else "10-K/10-Q")
    dt = parse_date_from_name(base)
    date_str = dt.strftime("%m-%d-%Y") if dt else "unknown"
    final_dir = FINAL_JSON_DIR / ticker
    final_dir.mkdir(parents=True, exist_ok=True)
    fname = f"{ticker}_{date_str}_{form}_FACTORS_DETAILED.json"
    return final_dir / fname

# ------------- parallel helpers -------------
_inflight = threading.BoundedSemaphore(MAX_IN_FLIGHT_OPENAI)

def needs_answers_file(ans_path: Path, expected_questions_hash: str) -> bool:
    if not ans_path.exists(): return True
    try:
        d = json.loads(ans_path.read_text(encoding="utf-8"))
        if not isinstance(d, dict):
            return True
        if d.get("questions_hash") != expected_questions_hash:
            return True
        ans = d.get("answers")
        if not isinstance(ans, list):
            return True
        if len(ans) == 0:
            return True
        return False
    except Exception:
        return True

def retry_call_extract(model: str, prompt: str, chunk_id: str, raw_debug_path: Path | None = None) -> Dict:
    assert CLIENT is not None, "OpenAI client not initialized"
    for attempt in range(1, MAX_RETRIES + 1):
        with _inflight:
            try:
                return call_extract(CLIENT, model, prompt, chunk_id, raw_debug_path=raw_debug_path)
            except Exception as e:
                msg = str(e).lower()
                fatal = any(x in msg for x in (
                    "invalid_api_key", "401", "unauthorized", "permission",
                    "model_not_found", "does not exist", "not found"
                ))
                if fatal or attempt == MAX_RETRIES:
                    raise
        sleep_s = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.25)
        time.sleep(sleep_s)

# ------------- per-file processing -------------
def process_report_from_file(html_path: Path, client: "OpenAI",
                             key_qs: List[Tuple[str, str]], q_hash: str):
    assert html_path.exists(), f"Missing: {html_path}"
    # ---- final-output-based skip ----
    final_path = expected_final_json_path(html_path)
    if final_path.exists():
        print(f"[skip-final-exists] {final_path.name} already exists, skipping {html_path.name}")
        return

    enc = safe_tokenizer()

    base = html_path.name
    stem = html_path.stem
    ticker = base.split("_")[0]
    form = "10-K" if "_10-K_" in base else ("10-Q" if "_10-Q_" in base else "10-K/10-Q")
    date_obj = parse_date_from_name(base)
    date_str = date_obj.strftime("%m-%d-%Y") if date_obj else "unknown"

    # All intermediates go to factors/intermediate/{TICKER}/
    run_dir = make_run_dir(html_path)
    # Filenames include the stem prefix to avoid collisions across multiple reports
    cleaned_path = run_dir / f"{stem}__CLEANED.html"
    index_path   = run_dir / f"{stem}__index.json"

    # 1) Clean + chunk (resume-safe)
    if not index_path.exists():
        raw_html = html_path.read_text(encoding="utf-8", errors="ignore")
        simplified = clean_html(raw_html)
        cleaned_path.write_text(simplified, encoding="utf-8")
        try:
            bt_ret = build_block_tree(simplified, max_node_words=256)
        except TypeError:
            bt_ret = build_block_tree(simplified)
        block_tree = bt_ret[0] if isinstance(bt_ret, tuple) else bt_ret

        CHUNK_BUDGET = 4200   # keep consistent with index fields
        chunks = pack_blocks_into_chunks(block_tree, budget_tokens=CHUNK_BUDGET, enc=enc, overlap_blocks=1)

        chunk_paths = []
        for i, chunk_html in enumerate(chunks, 1):
            p = run_dir / f"{stem}__chunk_{i:03d}.html"
            p.write_text(chunk_html, encoding="utf-8")
            chunk_paths.append(str(p))

        idx = {
            "source_file": str(html_path),
            "max_node_words": 256,
            "chunk_token_budget": CHUNK_BUDGET,
            "overlap_blocks": 1,
            "num_chunks": len(chunk_paths),
            "chunks": chunk_paths,
        }
        index_path.write_text(json.dumps(idx, indent=2), encoding="utf-8")
        print(f"[chunk] {base}: {len(chunk_paths)} chunks -> {run_dir}")

    # 2) Per-chunk extraction (parallel, resume-safe)
    idx = json.loads(index_path.read_text(encoding="utf-8"))
    todo = []
    for cp in idx.get("chunks", []):
        ap = Path(cp).with_suffix(".answers.json")  # stored under run_dir
        if needs_answers_file(ap, q_hash):
            todo.append((cp, ap))

    if todo:
        print(f"  extracting {len(todo)} chunks with {MAX_WORKERS_CHUNKS} workers…")
        def _task(cp: str, ap: Path):
            chunk_html = Path(cp).read_text(encoding="utf-8")
            prompt = build_prompt_for_chunk(chunk_html, key_qs)
            raw_path = ap.with_suffix(".raw.txt")
            data   = retry_call_extract(EXTRACT_MODEL, prompt, Path(cp).stem, raw_debug_path=raw_path)
            out = {
                "chunk_id": data.get("chunk_id", Path(cp).stem),
                "schema_version": SCHEMA_VERSION,
                "model": EXTRACT_MODEL,
                "questions_file": str(QUESTIONS_JSON),
                "questions_hash": q_hash,
                "answers": data.get("answers", [])
            }
            ap.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
            return ap.name

        with ThreadPoolExecutor(max_workers=MAX_WORKERS_CHUNKS) as ex:
            futures = [ex.submit(_task, cp, ap) for (cp, ap) in todo]
            for f in as_completed(futures):
                try:
                    wrote = f.result()
                    print(f"    ✓ {wrote}")
                except Exception as e:
                    print("    [chunk error]", e)

    # 3) Synthesis (serial per report)
    per_chunk = collect_per_chunk_answers_by_index(index_path)
    if not per_chunk:
        print(f"[warn] no per-chunk answers for {base}")
        return

    factors_map = aggregate_by_factor(per_chunk)
    keys = sorted(factors_map.keys())
    if not keys:
        print(f"[info] no factors with evidence for {base}")
        return

    def _default_synth_item(factor_key: str, rationale: str, conf: float = 0.1) -> Dict:
        return {
            "factor": factor_key,
            "detailed_summary": "",
            "impact": {"classification": "Unclear", "rationale": rationale, "confidence": conf},
        }

    items: List[Dict] = []
    pending: List[Tuple[List[str], str]] = []
    batch_i = 0
    for i in range(0, len(keys), MAX_FACTORS_PER_BATCH):
        batch_i += 1
        bk = keys[i:i+MAX_FACTORS_PER_BATCH]
        pending.append((bk, f"{batch_i:03d}"))

    while pending:
        bk, tag = pending.pop(0)
        payload = [{"factor": k,
                    "evidence": factors_map[k]["evidence"],
                    "notes":    factors_map[k]["notes"]} for k in bk]
        prompt = build_synth_prompt(ticker, form, payload)
        debug_path = run_dir / f"{stem}__synth_{tag}.raw.txt"
        try:
            data = call_synthesize(client, SYNTH_MODEL, prompt, raw_debug_path=debug_path)
        except Exception as e:
            # If this batch fails (often due to invalid/truncated JSON), split and retry.
            if len(bk) > 1:
                mid = max(1, len(bk) // 2)
                pending.insert(0, (bk[mid:], tag + "b"))
                pending.insert(0, (bk[:mid], tag + "a"))
                continue
            print(f"[warn] synth failed for {base} ({tag}): {e}")
            items.append(_default_synth_item(bk[0], "Synthesis failed; insufficient usable JSON.", 0.1))
            continue

        got = {it.get("factor", "") for it in data.get("items", [])}
        missing = [k for k in bk if k not in got]
        for m in missing:
            data["items"].append(_default_synth_item(m, "Insufficient evidence.", 0.2))
        items.extend(data["items"])
        time.sleep(PAUSE_BETWEEN_CALLS)

    final = {
        "source_file": str(html_path),
        "model": SYNTH_MODEL,
        "company_hint": ticker,
        "filing_hint": form,
        "report_date": date_str,
        "num_factors": len(items),
        "factors": items
    }

    # Only write the final result under FINAL_JSON_DIR; do not write back to sec_data/...
    final_dir = FINAL_JSON_DIR / ticker
    final_dir.mkdir(parents=True, exist_ok=True)
    clean_name = f"{ticker}_{date_str}_{form}_FACTORS_DETAILED.json"
    clean_out = final_dir / clean_name
    clean_out.write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[done] {clean_out} ({len(items)} factors)")

# ------------- MAIN -------------
if __name__ == "__main__":
    # Using local GPT OSS - no API key required
    assert BASE_DIR.exists(),       f"Base dir not found: {BASE_DIR}"
    assert QUESTIONS_JSON.exists(), f"Questions file not found: {QUESTIONS_JSON}"
    FINAL_JSON_DIR.mkdir(parents=True, exist_ok=True)
    INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

    # Load questions + client
    key_qs_text = QUESTIONS_JSON.read_text(encoding="utf-8")
    key_qs_hash = sha256_text(key_qs_text)
    key_qs_obj  = json.loads(key_qs_text)
    key_qs      = flatten_questions(key_qs_obj)

    # Singleton init (local GPT OSS at http://127.0.0.1:8000/v1)
    CLIENT = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")

    # Discover reports
    reports = find_reports(BASE_DIR,
                           tickers_filter=TICKERS_FILTER,
                           start_year=START_YEAR,
                           limit_total=MAX_REPORTS_TOTAL,
                           limit_per_ticker=MAX_REPORTS_PER_TICKER)

    print(f"Found {len(reports)} report(s). Running with {MAX_WORKERS_REPORTS} workers…")

    def _process_one_report(path: Path):
        try:
            process_report_from_file(path, CLIENT, key_qs, key_qs_hash)  # pass CLIENT
        except Exception as e:
            print(f"[error] {path.name}: {e}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS_REPORTS) as ex:
        futures = [ex.submit(_process_one_report, p) for p in reports]
        for _ in as_completed(futures):
            pass

    print("All done.")


Found 1 report(s). Running with 2 workers…
  extracting 2 chunks with 10 workers…
    ✓ AAL_10-K_2016-02-24_mda__chunk_002.answers.json
    ✓ AAL_10-K_2016-02-24_mda__chunk_012.answers.json
[fallback] Parsed synthesis from content (no tool call)
[repair] Parsed synthesis from content
[done] factors/mda_factors/AAL/AAL_02-24-2016_10-K_FACTORS_DETAILED.json (24 factors)
All done.


# Sentiment Re-Scoring

In [ ]:
# =========================================================================================
#  Parallel 5-Label Re-Scoring — Overwrite `impact.classification` in MD&A factor JSONs
#  Purpose:
#    Re-score factor sentiments to a strict 5-class scale (“very bad”, “bad”, “neutral”,
#    “good”, “very good”) using OpenAI tool-calls, and overwrite `impact.classification`
#    only where labels are missing/invalid. Fully parallel, resume-safe.
#
#  Key Features:
#  1) **Selective Overwrite** — Detects factors with missing/non-standard labels and only
#     re-scores those; reports already fully valid are skipped.
#  2) **Batch + Tool Call Scoring** — Packs up to `MAX_FACTORS_PER_CALL` factors per
#     request; returns strict tool-function JSON for deterministic parsing.
#  3) **Parallel Reports** — Processes multiple reports concurrently (`MAX_WORKERS_REPORTS`)
#     while globally capping OpenAI concurrency (`MAX_IN_FLIGHT_OPENAI`).
#  4) **Robustness** — Exponential backoff with retries for transient API errors; thread-safe
#     client creation; graceful per-file error handling.
#  5) **Non-Destructive Structure** — Only updates `impact.classification`; preserves all
#     other fields (e.g., `factor`, `detailed_summary`, `impact.rationale`, etc.).
#
#  Inputs / Outputs:
#    - Input root:  `factors/mda_factors/{TICKER}/*.json` (consolidated factor files)
#    - Output:      In-place updates to `impact.classification` (file content rewritten)
#
#  Usage:
#    export OPENAI_API_KEY=sk-...
#    python parallel_5label_rescore.py
#    # Optional: set TICKERS_FILTER / MAX_REPORTS_TOTAL in the config block
#
#  Configuration Highlights:
#    ALLOWED = ["very bad","bad","neutral","good","very good"]
#    MODEL   = "gpt-4o-2024-08-06"; TEMPERATURE = 0
#    MAX_FACTORS_PER_CALL = 40
#    MAX_WORKERS_REPORTS = 6; MAX_IN_FLIGHT_OPENAI = 8
#
#  Safety & Resume:
#    - Skips whole report if all factor labels already valid.
#    - Retries transient failures; partial progress preserved.
# =========================================================================================


import os, json, time, random, threading
from pathlib import Path
from typing import List, Dict, Any, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- CONFIG ---
INPUT_ROOT =  Path("factors/mda_factors")
TICKERS_FILTER = {"AAL"}
MAX_REPORTS_TOTAL = 1  # 1 file for quick testing; None = all

# Do not hardcode secrets. Ensure OPENAI_API_KEY is set in the environment before running.
# os.environ["OPENAI_API_KEY"] = "sk-..."  # intentionally removed

# Five allowed labels (lowercase exactly)
ALLOWED = ["very bad", "bad", "neutral", "good", "very good"]

# Model / batching
MODEL = "openai/gpt-oss-20b"
TEMPERATURE = 0
MAX_FACTORS_PER_CALL = 40  # if a report has more factors, we split into multiple tool calls

# Parallelism + resilience
MAX_WORKERS_REPORTS  = 1    # number of reports processed in parallel
MAX_IN_FLIGHT_OPENAI = 1    # global cap for concurrent OpenAI calls
MAX_RETRIES          = 5
BASE_BACKOFF         = 0.75 # seconds

# --- OpenAI client (created per thread for safety) ---
from openai import OpenAI
# Using local GPT OSS - no API key required

# ------------------- IO / discovery -------------------
def iter_report_files(root: Path, tickers_filter=None, limit=None):
    count = 0
    for ticker_dir in sorted(root.iterdir()):
        if not ticker_dir.is_dir():
            continue
        ticker = ticker_dir.name
        if tickers_filter and ticker not in tickers_filter:
            continue
        for p in sorted(ticker_dir.glob("*.json")):
            yield p
            count += 1
            if limit is not None and count >= limit:
                return

# ------------------- helpers -------------------
def is_valid_label(x: Any) -> bool:
    """Label must be a lowercase string in ALLOWED."""
    if not isinstance(x, str):
        return False
    return x.strip().lower() in ALLOWED

def get_current_label(f: Dict[str, Any]) -> str | None:
    imp = f.get("impact")
    if not isinstance(imp, dict):
        return None
    lab = imp.get("classification")
    if isinstance(lab, str):
        return lab.strip().lower()
    return None

# ------------------- prompt building -------------------
def build_batch_items(factors: List[Dict[str,Any]]) -> List[Dict[str,str]]:
    slim=[]
    for it in factors:
        slim.append({
            "factor": it.get("factor",""),
            "detailed_summary": it.get("detailed_summary",""),
            "rationale": (it.get("impact") or {}).get("rationale","")
        })
    return slim

def make_prompt(items_slim: List[Dict[str,str]]) -> str:
    return (
        "You are rating factor-level disclosures from an MD&A for likely stock-move direction.\n"
        "Assign exactly one label from this 5-class scale (lowercase exactly):\n"
        "  very bad, bad, neutral, good, very good\n\n"
        "Guidance:\n"
        "- Use ONLY the provided text (detailed_summary + rationale). No outside knowledge.\n"
        "- Clear negative effect on earnings/cash flows/risk → 'bad' or 'very bad' depending on severity.\n"
        "- Clear positive effect → 'good' or 'very good' depending on magnitude/clarity.\n"
        "- Balanced/unclear → 'neutral'.\n"
        "- Return ONLY via the tool function; no prose.\n\n"
        "FACTOR INPUTS:\n" + json.dumps(items_slim, ensure_ascii=False, indent=2)
    )

def make_tools():
    return [{
        "type": "function",
        "function": {
            "name": "score_factors",
            "description": "Return a 5-class sentiment label for each factor.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "factor": {"type": "string"},
                                "classification": {"type": "string", "enum": ["very bad","bad","neutral","good","very good"]}
                            },
                            "required": ["factor", "classification"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["items"],
                "additionalProperties": False
            }
        }
    }]

# ------------- OpenAI call w/ retries & cap -------------
_inflight = threading.BoundedSemaphore(MAX_IN_FLIGHT_OPENAI)

def call_openai_score(items: List[Dict[str,str]]) -> Dict[str,str]:
    """Return {factor -> classification} for one batch of items."""
    prompt = make_prompt(items)
    tools  = make_tools()

    for attempt in range(1, MAX_RETRIES+1):
        with _inflight:
            try:
                client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
                resp = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": "Return strict JSON tool output only."},
                        {"role": "user",   "content": prompt},
                    ],
                    tools=tools,
                    tool_choice={"type":"function","function":{"name":"score_factors"}},
                    temperature=TEMPERATURE,
                    max_tokens=3000
                )
                msg = resp.choices[0].message
                if not msg.tool_calls:
                    return {}
                args = json.loads(msg.tool_calls[0].function.arguments)
                mapping = {}
                for it in args.get("items", []):
                    fac = it.get("factor","")
                    lab = (it.get("classification","") or "").strip().lower()
                    if fac and lab in ALLOWED:
                        mapping[fac] = lab
                return mapping
            except Exception as e:
                # transient?
                s = str(e).lower()
                transient = any(x in s for x in ("rate", "429", "timeout", "temporarily", "503", "500", "bad gateway"))
                if not transient or attempt == MAX_RETRIES:
                    raise
        # backoff outside semaphore
        time.sleep(BASE_BACKOFF*(2**(attempt-1)) + random.uniform(0,0.25))

# ------------------ per-report scoring ------------------
def score_report_file(report_path: Path) -> Tuple[int,int,bool]:
    """
    Overwrite impact.classification for all factors in the given report,
    but only (re)score factors that are missing/invalid.
    Returns (updated_count, total_factors, skipped_whole_report).
    """
    data = json.loads(report_path.read_text(encoding="utf-8"))
    factors = data.get("factors", [])
    if not factors:
        return (0, 0, False)

    # -------- RESUME LOGIC (factor-level) --------
    # 1) Identify factors that need re-scoring: missing classification or classification not in ALLOWED
    needs_rescore_idx = []
    for idx, f in enumerate(factors):
        cur = get_current_label(f)
        if not is_valid_label(cur):
            needs_rescore_idx.append(idx)

    # 2) If all are already valid 5-class labels, skip the entire report
    if not needs_rescore_idx:
        return (0, len(factors), True)

    # 3) Batch-call only for factors that need re-scoring
    updated = 0
    mapping_total: Dict[str,str] = {}
    to_score = [factors[i] for i in needs_rescore_idx]
    for i in range(0, len(to_score), MAX_FACTORS_PER_CALL):
        chunk = to_score[i:i+MAX_FACTORS_PER_CALL]
        slim  = build_batch_items(chunk)
        batch_map = call_openai_score(slim)
        mapping_total.update(batch_map)

    # 4) Apply new labels (overwrite only the re-scored factors)
    for i in needs_rescore_idx:
        f = factors[i]
        k = f.get("factor","")
        if not k:
            continue
        new_label = mapping_total.get(k)
        if new_label and new_label in ALLOWED:
            if "impact" not in f or not isinstance(f["impact"], dict):
                f["impact"] = {}
            f["impact"]["classification"] = new_label
            updated += 1

    # 5) Persist to disk
    report_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return (updated, len(factors), False)

# -------------------- driver (parallel) --------------------
files = list(iter_report_files(
    INPUT_ROOT,
    tickers_filter=set(TICKERS_FILTER) if TICKERS_FILTER else None,
    limit=MAX_REPORTS_TOTAL
))
print(f"Found {len(files)} report JSON(s) to score.")
if not files:
    raise SystemExit("No files found under INPUT_ROOT.")

def _work(p: Path):
    try:
        upd, tot, skipped = score_report_file(p)
        if skipped:
            print(f"↷ {p.name}: skipped (already fully labeled) [{tot} factors]")
        else:
            print(f"✓ {p.name}: updated {upd}/{tot}")
        return upd, tot, skipped
    except Exception as e:
        print(f"[error] {p}: {e}")
        return 0, 0, False

updated_total = 0
factors_total = 0
skipped_reports = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS_REPORTS) as ex:
    futures = [ex.submit(_work, p) for p in files]
    for f in as_completed(futures):
        upd, tot, skipped = f.result()
        updated_total += upd
        factors_total += tot
        if skipped:
            skipped_reports += 1

print(f"\nDone. Updated {updated_total} of {factors_total} factor classifications. Skipped {skipped_reports} report(s).")


Found 1 report JSON(s) to score.


2026-02-15 16:37:48,833 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:37:48,835 - INFO - Retrying request to /chat/completions in 18.666000 seconds
2026-02-15 16:37:49,057 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-15 16:37:49,166 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:37:49,167 - INFO - Retrying request to /chat/completions in 17.544000 seconds


    ✓ AAL_10-K_2023-02-22_mda__chunk_001.answers.json


2026-02-15 16:37:49,871 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:37:49,873 - INFO - Retrying request to /chat/completions in 19.608000 seconds
2026-02-15 16:37:51,688 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-15 16:37:51,795 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:37:51,796 - INFO - Retrying request to /chat/completions in 18.454000 seconds


    ✓ AAL_10-K_2022-02-22_mda__chunk_002.answers.json


...

2026-02-15 16:40:42,275 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:42,347 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:42,347 - INFO - Retrying request to /chat/completions in 21.290000 seconds


    [chunk error] Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-SVHwoWltv2sUyK4nbj8UWV4H on tokens per min (TPM): Limit 30000, Used 30000, Requested 9080. Please try again in 18.16s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


2026-02-15 16:40:42,979 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:42,980 - INFO - Retrying request to /chat/completions in 18.726000 seconds
2026-02-15 16:40:49,333 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:49,405 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:49,405 - INFO - Retrying request to /chat/completions in 4.006000 seconds
2026-02-15 16:40:50,268 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:50,362 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:50,362 - INFO - Retrying request to /chat/completions in 4.258000 seconds


    [chunk error] Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-SVHwoWltv2sUyK4nbj8UWV4H on tokens per min (TPM): Limit 30000, Used 22732, Requested 8567. Please try again in 2.598s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


2026-02-15 16:40:51,227 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:51,317 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:51,318 - INFO - Retrying request to /chat/completions in 18.542000 seconds
2026-02-15 16:40:53,493 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:53,496 - INFO - Retrying request to /chat/completions in 17.678000 seconds
2026-02-15 16:40:57,771 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:57,774 - INFO - Retrying request to /chat/completions in 18.392000 seconds
2026-02-15 16:40:58,639 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-02-15 16:40:58,640 - INFO - Retrying request to /chat/completions in 18.862000 

# Event Study

In [ ]:
# Build df from factor JSONs + filing returns (required for Event Study below)
import json
import pandas as pd
from pathlib import Path

FACTORS_ROOT = Path("factors/mda_factors")
RETURNS_CSV   = Path("filing_returns_1m.csv")

rows = []
for ticker_dir in sorted(FACTORS_ROOT.iterdir()):
    if not ticker_dir.is_dir():
        continue
    ticker = ticker_dir.name
    for jpath in sorted(ticker_dir.glob("*_FACTORS_DETAILED.json")):
        try:
            data = json.loads(jpath.read_text(encoding="utf-8"))
            report_date = data.get("report_date")
            form = data.get("filing_hint", "10-K")
            for f in data.get("factors", []):
                imp = f.get("impact") or {}
                lab = imp.get("classification", "").strip().lower()
                if lab and lab in ["very bad", "bad", "neutral", "good", "very good"]:
                    rows.append({
                        "ticker": ticker,
                        "factor": f.get("factor", ""),
                        "classification": lab,
                        "report_date": report_date,
                        "form": form,
                    })
        except Exception as e:
            print(f"Skip {jpath.name}: {e}")

if not rows:
    raise SystemExit("No factor data found. Run factor extraction (Cell 3) first.")

factors_df = pd.DataFrame(rows)
factors_df["filing_date"] = pd.to_datetime(factors_df["report_date"], format="%m-%d-%Y", errors="coerce")
factors_df = factors_df.dropna(subset=["filing_date"])

returns = pd.read_csv(RETURNS_CSV)
returns["filing_date"] = pd.to_datetime(returns["filing_date"])
returns = returns.rename(columns={"filing_date": "filing_date_ret"})[["ticker", "form", "filing_date_ret", "ret_21d"]]

df = factors_df.merge(returns, on=["ticker", "form"], how="inner")
df["date_diff_days"] = (df["filing_date"] - df["filing_date_ret"]).dt.days.abs()
df = df[df["date_diff_days"] <= 3]
df = df.sort_values("date_diff_days").drop_duplicates(
    subset=["ticker", "factor", "filing_date", "classification"], keep="first"
).drop(columns=["date_diff_days"])
df = df.rename(columns={"ret_21d": "excess_21d"})
df = df[["filing_date", "factor", "classification", "excess_21d", "ticker"]]

print(f"Built df: {len(df)} rows, {df['ticker'].nunique()} tickers")

In [ ]:
# =========================================================================================
#  MD&A Factor Event Study — Tech-Relative Excess Returns (21D) + Parent/Subfactor + LS
#  Purpose:
#    Unified script to:
#      (1) load MD&A factor events using **21D tech-relative excess returns**,
#      (2) add parent/subfactor hierarchy (parent = text before first '.'),
#      (3) build monthly equal-weight Long/Short portfolios with robust fallbacks,
#      (4) compute cross-sectional coverage (by month, by factor, by ticker),
#      (5) summarize parent-level label counts/scores, and
#      (6) export CSVs and plots for both subfactor (child) and parent levels.
#
#  Return convention:
#    • Prefer column `excess_21d` if present (e.g., stock_21d − Tech ETF_21d over same window);
#      otherwise fall back to `ret_21d`.
#
#  Workflow:
#    • Input df must contain: filing_date, factor, classification, excess_21d (or ret_21d), ticker
#    • Child level = original `factor`; Parent level = text before first '.'
#    • Portfolio rules per month:
#         LS preferred (N per side: long=good/very good, short=bad/very bad)
#         Fallback L-only or S-only (2N); last-resort N-only; else return=0
#    • Outputs saved in ./factor_event_study/
#
#  Key Params:
#    ALLOWED_LABELS = ["very bad","bad","neutral","good","very good"]
#    LONG_LABELS    = {"good","very good"}; SHORT_LABELS = {"bad","very bad"}
#    n_per_side = 2; fallback_multiplier = 2
# =========================================================================================

from __future__ import annotations

import json
from pathlib import Path
from typing import Tuple, Set, Optional, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Constants / label sets
# -------------------------
ALLOWED_LABELS = ["very bad", "bad", "neutral", "good", "very good"]
LONG_LABELS: Set[str] = {"good", "very good"}
SHORT_LABELS: Set[str] = {"bad", "very bad"}
LABEL_TO_SCORE = {"very bad": -2, "bad": -1, "neutral": 0, "good": 1, "very good": 2}

# -------------------------
# Utilities
# -------------------------
def month_end_align(s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce")
    return dt.dt.to_period("M").dt.to_timestamp("M")

def add_parent_subfactor(
    df: pd.DataFrame, *, factor_col: str = "factor", parent_col: str = "parent", sub_col: str = "subfactor"
) -> pd.DataFrame:
    """
    Adds:
      parent   = text before the first '.' (or full factor if no dot)
      subfactor= original factor string (child level)
    """
    out = df.copy()
    fac = out[factor_col].astype(str)
    out[parent_col] = fac.str.split(".", n=1).str[0]
    out[sub_col] = fac
    return out

def _pick_side_values(
    grp: pd.DataFrame,
    label_col: str,
    ret_col: str,
    id_col: Optional[str],
    side_labels: Set[str],
    k: int,
) -> pd.Series:
    """Deterministic equal-weight selection of k names on one side."""
    side = grp[grp[label_col].isin(side_labels)].copy()
    if side.empty:
        return pd.Series(dtype=float)
    if (id_col is not None) and (id_col in side.columns):
        side = side.sort_values(id_col, kind="mergesort")  # deterministic
    else:
        side = side.sort_index(kind="mergesort")
    return pd.to_numeric(side[ret_col], errors="coerce").dropna().head(k)

# ------------------------------------------------------------
# Portfolio construction (generic) — LS preferred / fallback
# ------------------------------------------------------------
def compute_factor_monthly_with_fallback(
    df_events_ret: pd.DataFrame,
    *,
    n_per_side: int = 2,
    date_col: str = "filing_date",
    factor_col: str = "factor",  # can be 'factor' (child) OR 'parent'
    label_col: str = "classification",
    ret_col: str = "excess_21d",  # DEFAULT: tech-relative excess return
    id_col: Optional[str] = "ticker",
    fallback_multiplier: int = 2,  # use 2N for long-only / short-only fallback
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Preferred: LS with N per side. Fallback: L-only or S-only with 2N (equal-weight).
    Short-only return = -mean(shorts). If neither side has enough names, return = 0.

    Returns:
      perf_df  : [factor, month_end, mode, used_k_long, used_k_short, portfolio_return]
      ranking  : factor-level mean/std/T/tstat + mode coverage columns
      coverage : per-factor counts of months by mode
    """
    req = {date_col, factor_col, label_col, ret_col}
    missing = [c for c in req if c not in df_events_ret.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df_events_ret.copy()
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(ALLOWED_LABELS)]
    df[ret_col] = pd.to_numeric(df[ret_col], errors="coerce")
    df = df.dropna(subset=[ret_col])
    df["month_end"] = month_end_align(df[date_col])

    fallback_k = max(1, int(fallback_multiplier * n_per_side))

    rows: List[dict[str, Any]] = []
    for (f, m), grp in df.groupby([factor_col, "month_end"], sort=False):
        # 1) Try LS (N per side)
        long_vals_N = _pick_side_values(grp, label_col, ret_col, id_col, LONG_LABELS, n_per_side)
        short_vals_N = _pick_side_values(grp, label_col, ret_col, id_col, SHORT_LABELS, n_per_side)

        if (len(long_vals_N) == n_per_side) and (len(short_vals_N) == n_per_side):
            pr = float(long_vals_N.mean() - short_vals_N.mean())
            rows.append(
                dict(
                    factor=f,
                    month_end=m,
                    mode="LS",
                    used_k_long=n_per_side,
                    used_k_short=n_per_side,
                    portfolio_return=pr,
                )
            )
            continue

        # 2) Fallback L-only or S-only (2N)
        long_vals_FB = _pick_side_values(grp, label_col, ret_col, id_col, LONG_LABELS, fallback_k)
        short_vals_FB = _pick_side_values(grp, label_col, ret_col, id_col, SHORT_LABELS, fallback_k)

        candidates = []
        if len(long_vals_FB) == fallback_k:
            candidates.append(("L-only", fallback_k, 0, float(long_vals_FB.mean())))
        if len(short_vals_FB) == fallback_k:
            candidates.append(("S-only", 0, fallback_k, float(-short_vals_FB.mean())))

        if candidates:
            mode, kL, kS, pr = sorted(candidates, key=lambda x: (abs(x[3]), x[1] + x[2]), reverse=True)[0]
            rows.append(
                dict(factor=f, month_end=m, mode=mode, used_k_long=kL, used_k_short=kS, portfolio_return=pr)
            )
            continue

        # 3) Last-resort: N-only on one side
        c2 = []
        if len(long_vals_N) == n_per_side:
            c2.append(("L-only", n_per_side, 0, float(long_vals_N.mean())))
        if len(short_vals_N) == n_per_side:
            c2.append(("S-only", 0, n_per_side, float(-short_vals_N.mean())))
        if c2:
            mode, kL, kS, pr = sorted(c2, key=lambda x: (abs(x[3]), x[1] + x[2]), reverse=True)[0]
            rows.append(
                dict(factor=f, month_end=m, mode=mode, used_k_long=kL, used_k_short=kS, portfolio_return=pr)
            )
            continue

        # 4) No construction
        rows.append(dict(factor=f, month_end=m, mode="NA", used_k_long=0, used_k_short=0, portfolio_return=0.0))

    perf_df = pd.DataFrame(rows).sort_values(["factor", "month_end"]).reset_index(drop=True)

    # Ranking and coverage
    if perf_df.empty:
        ranking = pd.DataFrame(
            columns=[
                "factor",
                "mean",
                "std",
                "T",
                "tstat",
                "coverage_LS",
                "coverage_L_only",
                "coverage_S_only",
                "coverage_NA",
            ]
        )
        coverage = pd.DataFrame(columns=["factor", "LS", "L-only", "S-only", "NA", "total_months"])
    else:
        g = perf_df.groupby("factor")["portfolio_return"]
        ranking = g.agg(mean="mean", std="std", T="count").reset_index()
        denom = ranking["std"].replace(0.0, np.nan) / np.sqrt(np.maximum(ranking["T"], 1))
        ranking["tstat"] = ranking["mean"] / denom

        cov = perf_df.pivot_table(
            index="factor", columns="mode", values="portfolio_return", aggfunc="size", fill_value=0
        )
        for col in ["LS", "L-only", "S-only", "NA"]:
            if col not in cov.columns:
                cov[col] = 0
        cov = cov[["LS", "L-only", "S-only", "NA"]].reset_index()
        cov["total_months"] = cov[["LS", "L-only", "S-only", "NA"]].sum(axis=1)
        ranking = (
            ranking.merge(cov, on="factor", how="left")
            .rename(
                columns={"LS": "coverage_LS", "L-only": "coverage_L_only", "S-only": "coverage_S_only", "NA": "coverage_NA"}
            )
            .sort_values(["mean", "tstat"], ascending=[False, False])
            .reset_index(drop=True)
        )
        coverage = cov

    return perf_df, ranking, coverage

# ------------------------------------------------------------
# Cross-sectional coverage (factor×month and factor×ticker)
# ------------------------------------------------------------
def compute_cross_sectional_coverage(
    df_events: pd.DataFrame,
    *,
    date_col: str = "filing_date",
    factor_col: str = "factor",  # can be 'factor' (child) OR 'parent' OR 'subfactor'
    label_col: str = "classification",
    id_col: str = "ticker",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      cs_monthly: factor×month_end coverage (unique tickers, events, long/short counts)
      cs_totals : per-factor totals across the whole sample
      factor_ticker_counts: per-factor×ticker counts (events & months active, long/short)
    """
    req = {date_col, factor_col, label_col, id_col}
    missing = [c for c in req if c not in df_events.columns]
    if missing:
        raise ValueError(f"[CS coverage] Missing required columns: {missing}")

    df = df_events.copy()
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(ALLOWED_LABELS)]
    df["month_end"] = month_end_align(df[date_col])

    # Monthly cross-sectional coverage
    cs_monthly = (
        df.groupby([factor_col, "month_end"])
          .agg(
              n_events=(id_col, "size"),
              n_tickers=(id_col, "nunique"),
              n_long=(label_col, lambda s: s.isin(LONG_LABELS).sum()),
              n_short=(label_col, lambda s: s.isin(SHORT_LABELS).sum()),
          )
          .reset_index()
          .sort_values([factor_col, "month_end"])
          .rename(columns={factor_col: "factor"})
    )

    # Per-factor totals
    cs_totals = (
        df.groupby(factor_col)
          .agg(
              total_events=(id_col, "size"),
              total_unique_tickers=(id_col, "nunique"),
              months_active=("month_end", "nunique"),
              long_events=(label_col, lambda s: s.isin(LONG_LABELS).sum()),
              short_events=(label_col, lambda s: s.isin(SHORT_LABELS).sum()),
          )
          .reset_index()
          .rename(columns={factor_col: "factor"})
          .sort_values(["total_unique_tickers", "total_events"], ascending=[False, False])
    )

    # Per factor×ticker counts
    factor_ticker_counts = (
        df.groupby([factor_col, id_col])
          .agg(
              n_events=(id_col, "size"),
              months_active=("month_end", "nunique"),
              long_events=(label_col, lambda s: s.isin(LONG_LABELS).sum()),
              short_events=(label_col, lambda s: s.isin(SHORT_LABELS).sum()),
              first_date=(date_col, "min"),
              last_date=(date_col, "max"),
          )
          .reset_index()
          .rename(columns={factor_col: "factor"})
          .sort_values(["factor", "n_events", "months_active"], ascending=[True, False, False])
    )

    return cs_monthly, cs_totals, factor_ticker_counts

# ------------------------------------------------------------
# Parent-level label summaries (counts + score)
# ------------------------------------------------------------
def summarize_parent_label_scores(
    df: pd.DataFrame, *, parent_col: str = "parent", label_col: str = "classification", date_col: str = "filing_date"
):
    dd = df.copy()
    dd[label_col] = dd[label_col].astype(str).str.strip().str.lower()
    dd = dd[dd[label_col].isin(ALLOWED_LABELS)]
    dd["score"] = dd[label_col].map(LABEL_TO_SCORE)
    dd["month_end"] = month_end_align(dd[date_col])

    counts_overall = (
        dd.groupby([parent_col, label_col]).size().rename("count").reset_index()
        .pivot_table(index=parent_col, columns=label_col, values="count", fill_value=0)
        .reset_index()
    )
    counts_by_month = (
        dd.groupby([parent_col, "month_end", label_col]).size().rename("count").reset_index()
        .pivot_table(index=[parent_col, "month_end"], columns=label_col, values="count", fill_value=0)
        .reset_index()
    )
    score_by_month = dd.groupby([parent_col, "month_end"])["score"].mean().rename("avg_score").reset_index()
    score_overall = dd.groupby(parent_col)["score"].mean().rename("avg_score_overall").reset_index()

    # Ensure all label columns exist
    for lab in ALLOWED_LABELS:
        if lab not in counts_overall.columns:
            counts_overall[lab] = 0
        if lab not in counts_by_month.columns:
            counts_by_month[lab] = 0

    lbls = ALLOWED_LABELS
    counts_overall = counts_overall[[parent_col] + lbls]
    counts_by_month = counts_by_month[[parent_col, "month_end"] + lbls]

    return counts_overall, counts_by_month, score_by_month, score_overall

# ------------------------------------------------------------
# Plotting helpers
# ------------------------------------------------------------
def plot_top_factor_coverage_time_series(
    cs_monthly: pd.DataFrame,
    cs_totals: pd.DataFrame,
    *,
    top_n: int = 5,
    metric: str = "n_tickers",
    out_path: str = "factor_event_study/top_factor_coverage_timeseries.png",
) -> List[str]:
    """Time-series plot of monthly coverage for top factors by unique tickers."""
    assert metric in {"n_tickers", "n_events", "n_long", "n_short"}

    top_factors = cs_totals.nlargest(top_n, "total_unique_tickers")["factor"].tolist()
    sub = cs_monthly[cs_monthly["factor"].isin(top_factors)].copy()

    wide = (
        sub.pivot_table(index="month_end", columns="factor", values=metric, aggfunc="sum")
        .sort_index()
        .fillna(0.0)
    )

    plt.figure(figsize=(12, 6))
    for col in wide.columns:
        plt.plot(wide.index, wide[col], label=str(col))
    plt.title(f"Monthly {metric} coverage (Top {top_n} factors by unique tickers)")
    plt.xlabel("Month")
    plt.ylabel(metric)
    plt.legend(loc="best", ncol=2, frameon=False)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

    return top_factors

def plot_top_factor_ticker_hist(
    factor_ticker_counts: pd.DataFrame,
    top_factors: List[str],
    *,
    bins: int = 30,
    out_path: str = "factor_event_study/top_factor_ticker_hist.png",
) -> None:
    """Overlaid histogram of per-ticker coverage (n_events) for each top factor."""
    plt.figure(figsize=(12, 6))
    for f in top_factors:
        v = factor_ticker_counts.loc[factor_ticker_counts["factor"] == f, "n_events"].astype(int)
        if len(v) == 0:
            continue
        plt.hist(v, bins=bins, alpha=0.5, label=str(f))
    plt.title("Per-ticker coverage (n_events) — Top factors")
    plt.xlabel("n_events per ticker (factor-specific)")
    plt.ylabel("Count of tickers")
    plt.legend(loc="best", ncol=2, frameon=False)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

def top_tickers_by_factor(
    factor_ticker_counts: pd.DataFrame,
    *,
    top_k_per_factor: int = 10,
) -> pd.DataFrame:
    """For each factor, top tickers by n_events (tie-break: months_active)."""
    out = (
        factor_ticker_counts.sort_values(["factor", "n_events", "months_active"], ascending=[True, False, False])
        .groupby("factor")
        .head(top_k_per_factor)
        .reset_index(drop=True)
    )
    return out

def plot_parent_avg_subfactor_coverage(
    df_with_parent: pd.DataFrame,
    *,
    parent_col: str = "parent",
    sub_col: str = "subfactor",
    id_col: str = "ticker",
    out_path: str = "factor_event_study/parent_avg_subfactor_ticker_coverage.png",
) -> pd.DataFrame:
    """
    For each parent, compute each child's total unique tickers (across all time),
    take the average across its children, and plot a descending bar chart.
    """
    child_totals = df_with_parent.groupby(sub_col)[id_col].nunique().reset_index(name="child_unique_tickers")
    child_parent = df_with_parent[[parent_col, sub_col]].drop_duplicates()
    child_totals = child_totals.merge(child_parent, on=sub_col, how="left")

    parent_summary = (
        child_totals.groupby(parent_col)
        .agg(n_subfactors=(sub_col, "nunique"), avg_child_unique_tickers=("child_unique_tickers", "mean"))
        .reset_index()
        .sort_values("avg_child_unique_tickers", ascending=False)
    )

    plt.figure(figsize=(12, 6))
    plt.bar(parent_summary[parent_col].astype(str), parent_summary["avg_child_unique_tickers"])
    plt.title("Average subfactor ticker coverage by parent")
    plt.xlabel("Parent factor")
    plt.ylabel("Avg unique tickers per subfactor")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

    return parent_summary

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    # Expect df to be present in the environment OR load it here.
    # Required columns (case-insensitive resolver below):
    #    filing_date, factor, classification, excess_21d (or ret_21d), ticker
    try:
        df  # type: ignore[name-defined]
    except NameError:
        raise SystemExit(
            "Please define/load `df` with columns: "
            "[filing_date, factor, classification, excess_21d (or ret_21d), ticker]. "
            "For example: df = pd.read_csv('df.csv')"
        )

    OUTDIR = Path("factor_event_study")
    OUTDIR.mkdir(parents=True, exist_ok=True)

    # ---------- Column resolver (case-insensitive, with fallbacks) ----------
    cols = {c.lower(): c for c in df.columns}
    def resolve(*cands, default=None):
        for c in cands:
            cl = c.lower()
            if cl in cols:
                return cols[cl]
        return default

    date_col   = resolve("filing_date", "date", "report_date")
    factor_col = resolve("factor")
    label_col  = resolve("classification", "label", "impact_label", "class")
    # Prefer excess_21d if present; else ret_21d/return_21d/ret
    ret_in_col = resolve("excess_21d", "ret_21d", "return_21d", "ret")
    id_col     = resolve("ticker", "symbol", "cusip")

    if not all([date_col, factor_col, label_col, ret_in_col, id_col]):
        missing = [("filing_date", date_col), ("factor", factor_col), ("classification", label_col),
                   ("excess/ret", ret_in_col), ("ticker", id_col)]
        missing = [name for name, val in missing if val is None]
        raise SystemExit(f"Missing required columns: {missing}")

    # ---------- Enrich with parent/subfactor ----------
    # Normalize chosen return column to 'excess_21d' internally for consistency.
    df2 = df.rename(columns={
        date_col: "filing_date",
        factor_col: "factor",
        label_col: "classification",
        ret_in_col: "excess_21d",
        id_col: "ticker"
    })
    df2 = add_parent_subfactor(df2, factor_col="factor", parent_col="parent", sub_col="subfactor")

    # ======================= A) SUBFACTOR (child) LEVEL =======================
    perf_child, ranking_child, cov_modes_child = compute_factor_monthly_with_fallback(
        df2,
        n_per_side=2,
        date_col="filing_date",
        factor_col="subfactor",
        label_col="classification",
        ret_col="excess_21d",
        id_col="ticker",
        fallback_multiplier=2,
    )
    perf_child.to_csv(OUTDIR / "child_factor_monthly_with_fallback.csv", index=False)
    ranking_child.to_csv(OUTDIR / "child_factor_ranking_with_coverage.csv", index=False)
    cov_modes_child.to_csv(OUTDIR / "child_factor_mode_coverage.csv", index=False)

    cs_monthly_child, cs_totals_child, child_factor_ticker_counts = compute_cross_sectional_coverage(
        df2, date_col="filing_date", factor_col="subfactor", label_col="classification", id_col="ticker"
    )
    cs_monthly_child.to_csv(OUTDIR / "child_factor_monthly_cross_sectional_coverage.csv", index=False)
    cs_totals_child.to_csv(OUTDIR / "child_factor_cross_sectional_totals.csv", index=False)
    child_factor_ticker_counts.to_csv(OUTDIR / "child_factor_ticker_coverage_counts.csv", index=False)

    top_child = plot_top_factor_coverage_time_series(
        cs_monthly_child, cs_totals_child, top_n=5, metric="n_tickers",
        out_path=str(OUTDIR / "child_top_factor_coverage_timeseries.png"),
    )
    plot_top_factor_ticker_hist(
        child_factor_ticker_counts, top_child, bins=30,
        out_path=str(OUTDIR / "child_top_factor_ticker_hist.png"),
    )
    top_child_tickers = top_tickers_by_factor(child_factor_ticker_counts, top_k_per_factor=10)
    top_child_tickers.to_csv(OUTDIR / "child_factor_top_tickers_per_factor.csv", index=False)

    # =========================== B) PARENT LEVEL =============================
    perf_parent, ranking_parent, cov_modes_parent = compute_factor_monthly_with_fallback(
        df2,
        n_per_side=2,
        date_col="filing_date",
        factor_col="parent",
        label_col="classification",
        ret_col="excess_21d",
        id_col="ticker",
        fallback_multiplier=2,
    )
    perf_parent.to_csv(OUTDIR / "parent_factor_monthly_with_fallback.csv", index=False)
    ranking_parent.to_csv(OUTDIR / "parent_factor_ranking_with_coverage.csv", index=False)
    cov_modes_parent.to_csv(OUTDIR / "parent_factor_mode_coverage.csv", index=False)

    cs_monthly_parent, cs_totals_parent, parent_factor_ticker_counts = compute_cross_sectional_coverage(
        df2, date_col="filing_date", factor_col="parent", label_col="classification", id_col="ticker"
    )
    cs_monthly_parent.to_csv(OUTDIR / "parent_factor_monthly_cross_sectional_coverage.csv", index=False)
    cs_totals_parent.to_csv(OUTDIR / "parent_factor_cross_sectional_totals.csv", index=False)
    parent_factor_ticker_counts.to_csv(OUTDIR / "parent_factor_ticker_coverage_counts.csv", index=False)

    top_parent = plot_top_factor_coverage_time_series(
        cs_monthly_parent, cs_totals_parent, top_n=5, metric="n_tickers",
        out_path=str(OUTDIR / "parent_top_factor_coverage_timeseries.png"),
    )
    plot_top_factor_ticker_hist(
        parent_factor_ticker_counts, top_parent, bins=30,
        out_path=str(OUTDIR / "parent_top_factor_ticker_hist.png"),
    )
    top_parent_tickers = top_tickers_by_factor(parent_factor_ticker_counts, top_k_per_factor=10)
    top_parent_tickers.to_csv(OUTDIR / "parent_factor_top_tickers_per_factor.csv", index=False)

    # =================== C) Parent label counts / scores =====================
    counts_overall, counts_by_month, score_by_month, score_overall = summarize_parent_label_scores(
        df2, parent_col="parent", label_col="classification", date_col="filing_date"
    )
    counts_overall.to_csv(OUTDIR / "parent_factor_label_counts_overall.csv", index=False)
    counts_by_month.to_csv(OUTDIR / "parent_factor_label_counts_by_month.csv", index=False)
    score_by_month.to_csv(OUTDIR / "parent_factor_avg_score_by_month.csv", index=False)
    score_overall.to_csv(OUTDIR / "parent_factor_avg_score_overall.csv", index=False)

    # ================== D) Parent avg subfactor coverage plot =================
    parent_summary = plot_parent_avg_subfactor_coverage(
        df2, parent_col="parent", sub_col="subfactor", id_col="ticker",
        out_path=str(OUTDIR / "parent_avg_subfactor_ticker_coverage.png"),
    )
    parent_summary.to_csv(OUTDIR / "parent_avg_subfactor_ticker_coverage.csv", index=False)

    print("Saved outputs to:", OUTDIR.resolve())


# Majority-Vote

In [ ]:
# =========================================================================================
# Majority-Vote 5-Label Re-Scoring (MV-10) — Non-Destructive Copy, Robust Voting

## Key Features
# 1. **Multiple stochastic votes per factor**: Run `N` (default 10, configurable) independent labelings and aggregate with **majority voting**.
# 2. **Non-destructive workflow**: Writes outputs to `OUTPUT_ROOT`, mirroring `{ticker}/{filename}.json`, without overwriting originals in `INPUT_ROOT`.
# 3. **Batch + parallel with resilient retries**: Global concurrency cap, exponential backoff, and clear handling of transient vs. fatal errors.
# 4. **Strict 5-class schema (lowercase)**: `very bad`, `bad`, `neutral`, `good`, `very good`.
# 5. **Full audit trail**: Stores vote histogram and parameters in each factor’s `impact`:
#    - `impact.mv10_votes` — per-label counts
#    - `impact.mv10_params` — `{n_votes, temperature, model}`
# 6. **Tie-breaking policy**:
#    - Prefer the **original label** if it’s among the tied winners
#    - Else choose the label **closest to the mean score** (using a numeric mapping)
#    - Else default to **neutral**
# =========================================================================================

import os, json, time, random, threading
from pathlib import Path
from typing import List, Dict, Any, Tuple
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------- CONFIG ----------
INPUT_ROOT  = Path("factors/mda_factors")
OUTPUT_ROOT = Path("factors/mda_factors_mv10")  # <- NEW
TICKERS_FILTER    = None           # e.g., {"TT","GE"} or None for all tickers
MAX_REPORTS_TOTAL = None           # e.g., 50 for a dry-run; None = all

ALLOWED = ["very bad", "bad", "neutral", "good", "very good"]
LABEL_TO_SCORE = {"very bad": -2, "bad": -1, "neutral": 0, "good": 1, "very good": 2}
# Do not hardcode secrets. Ensure OPENAI_API_KEY is set in the environment before running.

# OpenAI / sampling
MODEL                 = "openai/gpt-oss-20b"
TEMPERATURE_VOTE      = 0.4 # 0.9          # higher temp for diversity
N_VOTES               = 3 # 10           # majority over 10 samples
MAX_FACTORS_PER_CALL  = 40           # batch size per call

# Parallelism + resilience
MAX_WORKERS_REPORTS   = 2 # 6
MAX_IN_FLIGHT_OPENAI  = 2 # 8
MAX_RETRIES           = 5
BASE_BACKOFF          = 2.0 # 0.75  # seconds
SKIP_IF_EXISTS        = True   # don't recompute if output JSON already exists

# ---------- OpenAI client ----------
from openai import OpenAI
# Using local GPT OSS - no API key required

# ---------- IO / discovery ----------
def iter_report_files(root: Path, tickers_filter=None, limit=None):
    count = 0
    for ticker_dir in sorted(root.iterdir()):
        if not ticker_dir.is_dir():
            continue
        ticker = ticker_dir.name
        if tickers_filter and ticker not in tickers_filter:
            continue
        for p in sorted(ticker_dir.glob("*.json")):
            yield ticker, p
            count += 1
            if limit is not None and count >= limit:
                return

def output_path_for(src_path: Path) -> Path:
    rel = src_path.relative_to(INPUT_ROOT)  # <ticker>/<file>.json
    out_path = OUTPUT_ROOT / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    return out_path

# ---------- prompt & tools ----------
def build_batch_items(factors: List[Dict[str,Any]]) -> List[Dict[str,str]]:
    slim=[]
    for it in factors:
        slim.append({
            "factor": it.get("factor",""),
            "detailed_summary": it.get("detailed_summary",""),
            "rationale": (it.get("impact") or {}).get("rationale","")
        })
    return slim

def make_prompt(items_slim: List[Dict[str,str]]) -> str:
    return (
        "You are rating factor-level disclosures from an MD&A for likely stock-move direction.\n"
        "Assign exactly one label from this 5-class scale (lowercase exactly):\n"
        "  very bad, bad, neutral, good, very good\n\n"
        "Guidance:\n"
        "- Use ONLY the provided text (detailed_summary + rationale). No outside knowledge.\n"
        "- Clear negative effect on earnings/cash flows/risk → 'bad' or 'very bad'.\n"
        "- Clear positive effect → 'good' or 'very good'.\n"
        "- Balanced/unclear → 'neutral'.\n"
        "- Return ONLY via the tool function; no prose.\n\n"
        "FACTOR INPUTS:\n" + json.dumps(items_slim, ensure_ascii=False, indent=2)
    )

def make_tools():
    return [{
        "type": "function",
        "function": {
            "name": "score_factors",
            "description": "Return a 5-class sentiment label for each factor.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "factor": {"type": "string"},
                                "classification": {
                                    "type": "string",
                                    "enum": ["very bad","bad","neutral","good","very good"]
                                }
                            },
                            "required": ["factor", "classification"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["items"],
                "additionalProperties": False
            }
        }
    }]

# ---------- OpenAI calling (with retries & cap) ----------
_inflight = threading.BoundedSemaphore(MAX_IN_FLIGHT_OPENAI)

def one_vote(items_slim: List[Dict[str,str]]) -> Dict[str,str]:
    """One stochastic vote over the batch; returns {factor -> label}."""
    prompt = make_prompt(items_slim)
    tools  = make_tools()

    for attempt in range(1, MAX_RETRIES+1):
        with _inflight:
            try:
                client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
                resp = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": "Return strict JSON tool output only."},
                        {"role": "user",   "content": prompt},
                    ],
                    tools=tools,
                    tool_choice={"type":"function","function":{"name":"score_factors"}},
                    temperature=TEMPERATURE_VOTE,
                    max_tokens=3000
                )
                msg = resp.choices[0].message
                if not msg.tool_calls:
                    return {}
                args = json.loads(msg.tool_calls[0].function.arguments)
                mapping = {}
                for it in args.get("items", []):
                    fac = it.get("factor","")
                    lab = (it.get("classification","") or "").strip().lower()
                    if fac and lab in ALLOWED:
                        mapping[fac] = lab
                return mapping
            except Exception as e:
                s = str(e).lower()
                transient = any(x in s for x in ("rate", "429", "timeout", "temporarily", "503", "500", "bad gateway", "overloaded", "dns", "reset by peer"))
                # 明确的致命类：认证/权限/模型不存在 -> 直接抛
                fatal = any(x in s for x in (
                    "invalid_api_key", "401", "unauthorized", "permission",
                    "model_not_found", "does not exist", "insufficient_quota"
                ))
                if fatal or attempt == MAX_RETRIES or not transient:
                    raise RuntimeError(f"OpenAI call failed (attempt {attempt}/{MAX_RETRIES}): {e}")
        time.sleep(BASE_BACKOFF*(2**(attempt-1)) + random.uniform(0,0.25))

def majority_label(counter: Counter, original: str|None=None) -> str:
    """Pick majority; break ties by (1) original if among ties, else (2) closest to mean score, else (3) neutral."""
    if not counter:
        return original if original in ALLOWED else "neutral"
    max_ct = max(counter.values())
    winners = [lab for lab, ct in counter.items() if ct == max_ct]
    if len(winners) == 1:
        return winners[0]
    # tie-break 1: prefer original if present
    if original in winners:
        return original
    # tie-break 2: closest to mean encoded score
    tot = sum(counter.values())
    mean_score = sum(LABEL_TO_SCORE[l]*c for l, c in counter.items()) / max(tot, 1)
    winners.sort(key=lambda l: (abs(LABEL_TO_SCORE[l] - mean_score),
                                ["neutral","good","bad","very good","very bad"].index(l)
                                if l in ["neutral","good","bad","very good","very bad"] else 99))
    return winners[0]

# ---------- per-report processing ----------
def rescore_report_file(src_path: Path) -> Tuple[int,int,str]:
    """
    Reads one input JSON, computes MV(10) labels, and writes a COPY to OUTPUT_ROOT.
    Returns (#updated, total_factors, out_filename).
    """
    dest_path = output_path_for(src_path)
    if dest_path.exists() and SKIP_IF_EXISTS:
        return (0, 0, f"skip (exists) → {dest_path.name}")
    print(f"开始处理 {src_path.name} ...", flush=True)

    data = json.loads(src_path.read_text(encoding="utf-8"))
    factors = data.get("factors", [])
    if not factors:
        # still write a passthrough copy to keep structure
        dest_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        return (0, 0, f"wrote empty → {dest_path.name}")

    # Prepare vote buckets per factor
    vote_counts: Dict[str, Counter] = defaultdict(Counter)

    # Batch through factors to keep prompts reasonable
    for i in range(0, len(factors), MAX_FACTORS_PER_CALL):
        chunk = factors[i:i+MAX_FACTORS_PER_CALL]
        items = build_batch_items(chunk)

        # do N_VOTES stochastic passes
        for _ in range(N_VOTES):
            vote = one_vote(items)
            for fac_key, lab in vote.items():
                vote_counts[fac_key][lab] += 1

    # Apply voted labels into a COPY of the data
    updated = 0
    for f in factors:
        k = f.get("factor","")
        if not k:
            continue
        original = ((f.get("impact") or {}).get("classification") or "").strip().lower()
        counts = vote_counts.get(k, Counter())
        if not counts:
            # no vote gathered; keep original
            continue
        new_label = majority_label(counts, original)
        if "impact" not in f or not isinstance(f["impact"], dict):
            f["impact"] = {}
        f["impact"]["classification"] = new_label                       # <- set voted label in COPY
        f["impact"]["mv10_votes"] = dict(counts)                        # <- keep histogram
        f["impact"]["mv10_params"] = {"n_votes": N_VOTES,
                                      "temperature": TEMPERATURE_VOTE,
                                      "model": MODEL}
        updated += int(new_label != original)

    # Write COPY (non-destructive)
    dest_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return (updated, len(factors), f"wrote → {dest_path.name}")

# ---------- driver ----------
def main():
    assert INPUT_ROOT.exists(), f"INPUT_ROOT not found: {INPUT_ROOT}"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    files = list(iter_report_files(
        INPUT_ROOT,
        tickers_filter=set(TICKERS_FILTER) if TICKERS_FILTER else None,
        limit=MAX_REPORTS_TOTAL
    ))
    print(f"Found {len(files)} report JSON(s). Writing to {OUTPUT_ROOT} (MV10; temp={TEMPERATURE_VOTE}).")

    updated_total = 0
    factors_total = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_REPORTS) as ex:
        futs = [ex.submit(rescore_report_file, p) for _, p in files]
        for f in as_completed(futs):
            try:
                upd, tot, msg = f.result()
                print(("✓ " if upd or ("wrote" in msg) else "• ") + msg)
                updated_total += upd
                factors_total += tot
            except Exception as e:
                print(f"✗ failed → {e}")

    print(f"\nDone. Majority-voted {updated_total} updated labels over {factors_total} factors.")
    print(f"Outputs in: {OUTPUT_ROOT}")

if __name__ == "__main__":
    main()

